# Week 8 (CS 418 @ UIC)

In [ ]:
import matplotlib
matplotlib.use("Agg")

In [ ]:
# Shared setup for every figure in this deck.
import numpy as np
import polars as pl
from plotnine import *

ORANGE = "#f9461c"

<!-- slide 1 -->
# Week 8 Slide Deck

<h2>Linear Regression; Intro to Classification</h2>

Jack Bandy
2026

<!-- slide 2 -->
# Week 8, Day 1

CS 418 · Week 8 · 🟠 Roosevelt 🟠

<!-- slide 3 -->
## Map

- Week 8: Roosevelt
- Day 1
    - Intro to Modeling
    - Loss Functions
    - Overfitting
    - Simple Linear Regression
    - LINE Criteria and Practice
    - Multiple Linear Regression

![](../assets/orange-line-map-stops/stop09-roosevelt.svg)

<!-- slide 4 -->
# Intro to Modeling

<!-- slide 5 -->
## The Data Science Lifecycle

<img src="../assets/lifecycle/ds-lifecycle-v3.svg" style="max-height: 380px;" alt="The data science lifecycle with the 'understand the world' stage highlighted: ask a question, obtain data, understand the data, and understand the world, connected by arrows that loop back as well as forward, with 'reports, decisions, solutions' as an output.">

Review: weeks 1–3 asked questions and obtained data; weeks 4–6 were about understanding the data (EDA, visualization); week 7 started inference. Models are how we move from describing *this* data to saying something about the world it came from.

<!-- slide 6 -->
## What is a model?

Sources: Data 100 course notes, UC Berkeley, ["Modeling & SLR"](https://ds100.org/course-notes/modeling-slr); Lau, Gonzalez, and Nolan, [*Learning Data Science*](https://learningds.org/ch/04/modeling_intro.html), ch. 4.

- "A model is an idealized representation of a system. A system is a set of principles or procedures according to which something functions." (Data 100)
- "We use models to approximate the signal, with the simplest of these being the constant model, where the signal is approximated by a single number, like the mean or median." (Lau, Gonzalez, and Nolan)
- We **build** / **fit** / **train** a model by estimating parameters from data
    - then use it to predict or to explain a phenomenon

<!-- slide 7 -->
## George Box

> Essentially, all models are wrong, but some are useful.

George E. P. Box, statistician (1919–2013)

George E. P. Box and Norman R. Draper, *Empirical Model-Building and Response Surfaces* (Wiley, 1987), p. 424.

<!-- slide 8 -->
## Physical vs. Statistical Models: Car MPG

How many miles per gallon will a car get?

**Physical (mechanistic) model**

- Based on how the system works
- braking, air resistance, rolling resistance, etc.
- can use equations based on real physics, real measurements, "laws"
    - e.g. mass, frontal area, drag coefficient, etc.

**Statistical model**

- Start from data about many cars.
    - e.g. the classic auto MPG dataset has 398 cars
- no physics needed
- For example, we could predict MPG based on year manufactured, brand, seats, etc.

<!-- slide 9 -->
## Mechanistic or Statistical?

Which kind of model is each one? Why? Discuss with elbow neighbor(s).

- **A.** Car mpg based on the year the car was manufactured
- **B.** Car mpg based on the car's weight
- **C.** Chicago temperature based on the day of the year (1-365)
- **D.** Chicago temperature based on tilt of Earth's axis

Sources: R. Quinlan, [Auto MPG](https://doi.org/10.24432/C5859H) [Dataset], UCI Machine Learning Repository (1993), whose variables include `model_year` and `weight`; NASA Space Place, ["What Causes the Seasons?"](https://spaceplace.nasa.gov/seasons/en/).

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

A: statistical. The model year does not burn fuel; it stands in for changes in engines and regulation, so the relationship comes from data. B: either. Fitting mpg to weight across cars is statistical, but physics also says mass matters (MacKay: energy goes into braking, and kinetic energy is ½mv²), so a mechanistic model would include it too. C: statistical. The day of the year is a stand-in for the season; you would fit a seasonal curve to past temperatures. D: mechanistic, built from the physics of sunlight. Good follow-up: which input actually matters? NASA: "Earth's tilted axis causes the seasons," and in the Northern Hemisphere "we are having winter when Earth is closest to the Sun." The change in distance "does not make much difference to our weather." A mechanistic model makes you say which inputs matter and why.

<!-- slide 10 -->
# Loss Functions

<!-- slide 11 -->
## Setup: Cook County Home Sales, 2025

Back to our running example: what does a home cost in Cook County?

In [ ]:
sales = pl.read_csv("../../datasets/cook-county-home-sales-2025/cook-county-home-sales-2025.csv",
                    ignore_errors=True)
homes = sales.filter(
    (pl.col("property_group") == "Single-Family")   # condos have no sqft
    & (pl.col("is_multisale") == False)             # one price per home
    & (pl.col("sale_price") > 10_000)               # drop transfers
    & pl.col("building_sqft").is_not_null()
)
homes.height

<!-- slide 12 -->
## The Parameter to Estimate: $\theta$

Source: Lau, Gonzalez, and Nolan, *Learning Data Science*, ["The Constant Model"](https://learningds.org/ch/04/modeling_simple.html) (§4.1) and ["Minimizing Loss"](https://learningds.org/ch/04/modeling_loss_functions.html) (§4.2).

- In modeling, $\theta$ is often referred to as a **parameter**.
- $\hat{\theta}$ ("theta hat") is the **estimate**, computed from data.

<!-- slide 13 -->
## Potential options for $\hat{\theta}$

Adapted from Lau, Gonzalez, and Nolan, ["The Constant Model"](https://learningds.org/ch/04/modeling_simple.html), *Learning Data Science*, §4.1, which compares the mode, median, and mean of bus lateness.

Options to choose for $\hat{\theta}$:

- the price of one randomly selected sale
- the mode of the prices
- the mean of the prices
- the median of the prices

. . .

In [ ]:
prices = homes["sale_price"]
print(f"mean = ${prices.mean():,.0f}   median = ${prices.median():,.0f}")

Which is better here?

<!-- slide 14 -->
## Loss Functions

Source: Lau, Gonzalez, and Nolan, ["Minimizing Loss"](https://learningds.org/ch/04/modeling_loss_functions.html), *Learning Data Science*, §4.2.

"A loss function ... takes in $\theta$ and a data value $y$. It outputs a single number, the loss, that measures how far away $\theta$ is from $y$."

- Loss for one point: $\ell(\theta, y)$
    - Lower is better.
- Average loss over all $n$ points:

$$L(\theta, \mathbf{y}) = \frac{1}{n} \sum_{i=1}^{n} \ell(\theta, y_i)$$

- **Fitting** a model: find $\hat{\theta}$ with the smallest average loss.

<!-- slide 15 -->
## Squared Loss vs. Absolute Loss (vs. ...)

Sources: Lau, Gonzalez, and Nolan, *Learning Data Science*, [§4.2](https://learningds.org/ch/04/modeling_loss_functions.html), [§19.4](https://learningds.org/ch/19/class_loss.html), and [§20.2](https://learningds.org/ch/20/gd_example.html); Data 100 course notes, UC Berkeley, ["Constant Model, Loss, and Transformations"](https://ds100.org/course-notes/loss-transformations).

- **Squared (L2)**: $(y - \theta)^2$, averaged = MSE
- **Absolute (L1)**: $|y - \theta|$, averaged = MAE
- Squared loss punishes big misses more.
- Others: Huber (a blend); log loss for classification (Day 2).
- Do outliers matter? Are all errors equally costly?
<img class="figure" src="../assets/week8/loss-functions.svg" alt="Two loss curves plotted against the error y minus theta, from minus 3 to 3. The absolute loss is a V shape reaching 3 at either end; the squared loss is a parabola that is below the V between minus 1 and 1 and climbs to 9 at either end.">

<!-- slide 16 -->
# Overfitting

<!-- slide 17 -->
## Which model would you choose?

Idea after Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1, which fits polynomials of degree 1 to 12 to one data set. Data: 14 Cook County single-family sales, 2025.

Which of the following models would you choose to represent this data? Why?

<img data-id="overfit-fig" src="../assets/week8/overfit-data.svg" alt="Three identical panels of the same 14 home sales, building square footage against sale price, with no model drawn yet.">

<!-- slide 18 -->
## Which model would you choose?

Idea after Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1, which fits polynomials of degree 1 to 12 to one data set. Data: 14 Cook County single-family sales, 2025.

Which of the following models would you choose to represent this data? Why?

<img data-id="overfit-fig" src="../assets/week8/overfit-a.svg" alt="The same three panels of 14 home sales. Panel A now has a flat horizontal line; panels B and C still show only the points.">

<!-- slide 19 -->
## Which model would you choose?

Idea after Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1, which fits polynomials of degree 1 to 12 to one data set. Data: 14 Cook County single-family sales, 2025.

Which of the following models would you choose to represent this data? Why?

<img data-id="overfit-fig" src="../assets/week8/overfit-ab.svg" alt="The same three panels of 14 home sales. A: a flat horizontal line. B: a straight line rising with square footage. Panel C still shows only the points.">

<!-- slide 20 -->
## Which model would you choose?

Idea after Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1, which fits polynomials of degree 1 to 12 to one data set. Data: 14 Cook County single-family sales, 2025.

Which of the following models would you choose to represent this data? Why?

<img data-id="overfit-fig" src="../assets/week8/overfit-abc.svg" alt="Three panels of the same 14 home sales, building square footage against sale price. A: a flat horizontal line. B: a straight line rising with square footage. C: a wildly wiggling curve that passes through nearly every point and shoots off the chart at the edges.">

<!-- slide 21 -->
## Discuss with your elbow neighbor(s)

Which model would you choose to represent this data? Why?

<img data-id="overfit-fig" class="figure" src="../assets/week8/overfit-abc.svg" alt="Three panels of the same 14 home sales, building square footage against sale price. A: a flat horizontal line. B: a straight line rising with square footage. C: a wildly wiggling curve that passes through nearly every point and shoots off the chart at the edges.">

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

<!-- slide 22 -->
## Underfitting and Overfitting

Source: Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1.

<img data-id="overfit-fig" src="../assets/week8/overfit-abc-labeled.svg" alt="The same three panels, now labeled. A, underfit: one number for every home. B, just right: a line follows the trend. C, overfit: chases the noise in 14 sales.">

"Overfitting happens when the model follows the data too closely and picks up the variability in the random noise in the outcome. When this happens, new observations are not well-predicted."

<!-- slide 23 -->
## Summary (So Far)

Source: Data 100 course notes, UC Berkeley, ["Modeling & SLR"](https://ds100.org/course-notes/modeling-slr), which calls this "the modeling process": choose a model, choose a loss function, fit the model, evaluate model performance.

1. **Define the model**: a simplified representation of the world.
    - Use domain knowledge
    - Try to keep it simple (for now)
2. **Define the loss function**: how well one instance of the model "fits" the data (e.g. squared or absolute loss, averaged over the data).
3. **Minimize the loss**: find $\hat{\theta}$, parameter values with smallest loss.

<!-- slide 24 -->
# Simple Linear Regression

<!-- slide 25 -->
## Linear Models

Source: Lau, Gonzalez, and Nolan, ["Simple Linear Model"](https://learningds.org/ch/15/linear_simple.html) and ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §15.1 and §16.1.

"We want to use information from a second feature to make a better model than the constant model."

- Constant model: $y \approx \theta$
- Simple linear model: $y \approx \theta_0 + \theta_1 x$
- "Linear" means linear in the parameters $\theta$.
    - "Polynomials are still considered linear models." (e.g. Model C in the overfitting example was a linear model!)

<!-- slide 26 -->
## Example: House Cost

- Correlation matrix for Cook County home sales (week 5)
- Goal is to predict **price**
- Building square footage strongly correlated with price
- Recall the "cone shape"
    - (the scatter fans out for larger homes)

<img class="figure" src="../assets/viz-basics/correlation-matrix.svg" alt="Seven-by-seven correlation matrix of Cook County home sale variables, shaded orange for positive and blue-grey for negative. Building square footage correlates 0.77 with rooms and 0.69 with price; miles from the Loop correlates only minus 0.08 with price.">

Data: Cook County single-family sales, 2025.

<!-- slide 27 -->
## Vocabulary for Simple Linear Regression

Source: Lau, Gonzalez, and Nolan, *Learning Data Science*, [§15.1](https://learningds.org/ch/15/linear_simple.html).

Simple linear regression models a linear relationship between two variables.

- The variable being modeled or predicted is called the **response** (or outcome) variable.
- The variable used to predict the response is called the **predictor** (or explanatory) variable.
- Lau, Gonzalez, and Nolan recommend avoiding "independent/dependent": these names "connote a causal relationship."
- One predictor gives a line, $\hat{y} = \theta_0 + \theta_1 x$
    - intercept $\theta_0$: the predicted value of $y$ when $x$ is zero
    - slope $\theta_1$: how much the predicted value of $y$ changes for every one-unit increase in $x$

![](../assets/week8/advertising-sales.svg)

<!-- slide 28 -->
## Least Squares: Pick the Line with the Smallest Loss

Source: Lau, Gonzalez, and Nolan, *Learning Data Science*, [§15.1](https://learningds.org/ch/15/linear_simple.html) and [§15.3](https://learningds.org/ch/15/linear_simple_fit.html).

Same three steps as before, now with two parameters:

1. Model: $\hat{y}_i = \theta_0 + \theta_1 x_i$
2. Loss: average squared **residual** (MSE)

$$L(\theta_0, \theta_1) = \frac{1}{n} \sum_{i=1}^{n} \big(y_i - (\theta_0 + \theta_1 x_i)\big)^2$$

3. Minimize: take partial derivatives with respect to $\theta_0$ and $\theta_1$, set them to zero, solve.
- The errors are measured **vertically**, from each point to the line.

<!-- slide 29 -->
## Correlation Coefficient: Recap from Week 5

The correlation coefficient $r$ measures the strength and direction of a linear relationship.

- $r$ always falls between $-1$ and $+1$.
- sign gives the direction: positive rises, negative falls.
- $|r|$ closer to 1: points hug the line.
- $r$ near 0 means no linear pattern (others may still exist)

![](../assets/week8/correlation-strengths.svg)

<p class="code-caption" style="margin-top:0.5em;">Same idea at four strengths</p>

<!-- slide 30 -->
## The Least Squares Solution

Sources: Lau, Gonzalez, and Nolan, ["Fitting the Simple Linear Model"](https://learningds.org/ch/15/linear_simple_fit.html), *Learning Data Science*, §15.3; Data 100 course notes, UC Berkeley, ["Modeling & SLR"](https://ds100.org/course-notes/modeling-slr).

Solving the minimization gives:

$$\hat{\theta}_1 = r \, \frac{s_y}{s_x} \qquad \hat{\theta}_0 = \bar{y} - \hat{\theta}_1 \bar{x}$$

- $\bar{x}, \bar{y}$: sample means. $s_x, s_y$: sample standard deviations. $r$: correlation coefficient.
- The slope is the correlation, converted into the units of $y$ per unit of $x$.
- The line always passes through the point $(\bar{x}, \bar{y})$.

<!-- slide 31 -->
## Fitting the House Model

In [ ]:
# Pull the two columns out as NumPy arrays: x = predictor, y = response
x = homes["building_sqft"].to_numpy()
y = homes["sale_price"].to_numpy()

# corrcoef returns a 2x2 matrix; the off-diagonal [0, 1] entry is r(x, y)
r = np.corrcoef(x, y)[0, 1]
# Slope: r rescaled into dollars per square foot (ddof=1: sample SD)
theta_1 = r * y.std(ddof=1) / x.std(ddof=1)
# Intercept: makes the line pass through the point (mean of x, mean of y)
theta_0 = y.mean() - theta_1 * x.mean()
print(f"r = {r:.3f}   slope = {theta_1:,.0f}   intercept = {theta_0:,.0f}")

. . .

- How would you interpret the slope? The intercept?

Slope: each additional square foot is associated with roughly $426 more in sale price, on average. Intercept: the predicted price of a 0 sq ft home is negative, which is meaningless; the intercept is an extrapolation far outside the data. "Associated with": this is not a causal claim (Week 7).

<!-- slide 32 -->
## The Fitted Line

In [ ]:
(ggplot(homes, aes("building_sqft", "sale_price"))
 + geom_point(color="#777777", alpha=0.08, size=0.8, raster=True)
 + geom_abline(intercept=theta_0, slope=theta_1, color=ORANGE, size=1.2)
 + scale_y_continuous(limits=(0, 2_000_000),
                      labels=lambda v: [f"${x/1e6:g}M" for x in v])
 + scale_x_continuous(limits=(0, 6000))
 + labs(x="Building square footage", y="Sale price")
 + theme_minimal())

<!-- slide 33 -->
# LINE Criteria

<!-- slide 34 -->
## Assumptions / Criteria

Source: Mine Çetinkaya-Rundel and Johanna Hardin, ["Inference for Linear Regression with a Single Predictor"](https://openintro-ims.netlify.app/inf-model-slr.html), §24.6 of *Introduction to Modern Statistics*, 2nd ed. (OpenIntro).

"When fitting a least squares line, we generally require the following" (the **LINE** mnemonic):

- **Linearity.** "The data should show a linear trend."
- **Independent observations.** "Be cautious about applying regression to data that are sequential observations in time such as a stock price each day."
- **Nearly normal residuals.** "Generally, the residuals should be nearly normal."
- **Equal variability** (homoscedasticity). "The variability of points around the least squares line remains roughly constant."
- "The linearity, normality, and equality of variance conditions usually can be assessed through residual plots," using residuals $y_i - \hat{y}_i$.

<!-- slide 35 -->
# Practice: LINE Criteria

<!-- slide 36 -->
## How to Practice

Source: Çetinkaya-Rundel and Hardin, ["Inference for Linear Regression with a Single Predictor"](https://openintro-ims.netlify.app/inf-model-slr.html), *Introduction to Modern Statistics*, 2nd ed., §24.6. Each example's data source is printed under its plots.

Each example checks one criterion.

- **Linearity**: residuals vs. $x$ should show no leftover pattern.
- **Independence**: residuals in the order collected should show no runs or waves.
- **Nearly normal residuals**: the histogram should look like a bell.
- **Equal variability**: residuals vs. fitted values should be an even band.

##

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-b-v0.svg" alt="Linearity practice: a scatter plot of synthetic data B, a quadratic curve, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-b-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows U-shaped residuals.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-b-v2.svg" alt="Linearity practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-c-v0.svg" alt="Linearity practice: a scatter plot of synthetic data A, a straight line plus normal noise, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-c-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows a patternless band of residuals.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-c-v2.svg" alt="Linearity practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-a-v0.svg" alt="Linearity practice: a scatter plot of 300 Cook County home sales from 2025, log building square footage against log sale price, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-a-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows no curve left in the residuals.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-a-v2.svg" alt="Linearity practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-c-v0.svg" alt="Linearity practice: a scatter plot of synthetic data C, a saturating curve, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-c-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows an upside-down U in the residuals.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-c-v2.svg" alt="Linearity practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-a-v0.svg" alt="Linearity practice: a scatter plot of CTA rail boardings by year, 1988 to 2025, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-a-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows residuals that dip, climb, then crash in 2020.">

<img src="../assets/linear-model-criteria-assumptions/linearity-negative-example-a-v2.svg" alt="Linearity practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-b-v0.svg" alt="Linearity practice: a scatter plot of Olympic men's against women's 100 m winning times at the same Games, 1928 to 2024, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-b-v1.svg" alt="Linearity practice: the same data with its least-squares line, next to a plot of residuals against x, which shows residuals scattered evenly around zero.">

<img src="../assets/linear-model-criteria-assumptions/linearity-positive-example-b-v2.svg" alt="Linearity practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-c-v0.svg" alt="Independent observations practice: a scatter plot of synthetic data D, with independent normal errors, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-c-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows jagged residuals with no runs.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-c-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-c-v0.svg" alt="Independent observations practice: a scatter plot of synthetic data E, whose errors each carry over 85 percent of the previous one, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-c-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows slow drifts, with each residual near the last.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-c-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-a-v0.svg" alt="Independent observations practice: a scatter plot of monthly airline passengers, 1949 to 1960, on a log scale, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-a-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows a repeating yearly wave.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-a-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-a-v0.svg" alt="Independent observations practice: a scatter plot of 300 Cook County home sales from 2025, log-log, ordered by sale date, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-a-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows no runs or waves across the year.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-a-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-b-v0.svg" alt="Independent observations practice: a scatter plot of Chicago buildings with 10 or more stories, stories against units, ordered by year built, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-b-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows neighbors in time that don't track each other.">

<img src="../assets/linear-model-criteria-assumptions/independence-positive-example-b-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-b-v0.svg" alt="Independent observations practice: a scatter plot of weekday entries at the CTA UIC-Halsted station in 2025, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-b-v1.svg" alt="Independent observations practice: the same data with its least-squares line, next to a plot of residuals in the order the data were collected, which shows long runs above and below zero that line up with the academic calendar.">

<img src="../assets/linear-model-criteria-assumptions/independence-negative-example-b-v2.svg" alt="Independent observations practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-c-v0.svg" alt="Nearly normal residuals practice: a scatter plot of synthetic data H, with exponential errors shifted to mean zero, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-c-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows right skew: a pile-up just below zero and a long right tail.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-c-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-b-v0.svg" alt="Nearly normal residuals practice: a scatter plot of synthetic data F, 200 points with normal errors, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-b-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows a bell shape.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-b-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-a-v0.svg" alt="Nearly normal residuals practice: a scatter plot of Olympic men's 100 m winning times by year, 1948 to 2024, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-a-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows a rough bell shape from only 20 points.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-a-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-a-v0.svg" alt="Nearly normal residuals practice: a scatter plot of IMDb's 1,000 most-rated movies, release year against number of ratings, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-a-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows strong right skew: a wall just below zero and a long right tail.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-a-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-c-v0.svg" alt="Nearly normal residuals practice: a scatter plot of synthetic data G, 400 points with normal errors, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-c-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows a bell shape.">

<img src="../assets/linear-model-criteria-assumptions/normality-positive-example-c-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-b-v0.svg" alt="Nearly normal residuals practice: a scatter plot of US presidential elections, year against each candidate's vote share, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-b-v1.svg" alt="Nearly normal residuals practice: the same data with its least-squares line, next to a histogram of the residuals with a normal curve overlaid, which shows two humps, with almost nothing near zero.">

<img src="../assets/linear-model-criteria-assumptions/normality-negative-example-b-v2.svg" alt="Nearly normal residuals practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-b-v0.svg" alt="Equal variability practice: a scatter plot of synthetic data I, 150 points with constant-variance errors, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-b-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows an even band of residuals.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-b-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-c-v0.svg" alt="Equal variability practice: a scatter plot of synthetic data K, whose error spread shrinks as x grows, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-c-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows residuals whose spread changes steadily with the fitted value.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-c-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-a-v0.svg" alt="Equal variability practice: a scatter plot of CTA 'L' stations in 2025, average weekday against weekend and holiday entries, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-a-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows a tight band for small stations and a wide fan for the busiest.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-a-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-a-v0.svg" alt="Equal variability practice: a scatter plot of IMDb movies, runtime against average rating, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-a-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows a roughly even band of residuals.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-a-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Meets criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-b-v0.svg" alt="Equal variability practice: a scatter plot of monthly airline passengers, 1949 to 1960, on the raw scale, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-b-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows residuals that fan out as traffic grows.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-negative-example-b-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Does not meet criteria' in orange.">

##

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-c-v0.svg" alt="Equal variability practice: a scatter plot of synthetic data J, 60 points with constant-variance errors, with no line or diagnostic yet.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-c-v1.svg" alt="Equal variability practice: the same data with its least-squares line, next to a plot of residuals against fitted values, which shows an even band of residuals.">

<img src="../assets/linear-model-criteria-assumptions/equal-variability-positive-example-c-v2.svg" alt="Equal variability practice: the same two panels, stamped 'Meets criteria' in orange.">

<!-- slide 37 -->
## Residuals for the House Model

In [ ]:
predicted = theta_0 + theta_1 * pl.col("building_sqft")
homes = homes.with_columns((pl.col("sale_price") - predicted).alias("residual"))
(ggplot(homes, aes("building_sqft", "residual"))
 + geom_point(color="#777777", alpha=0.08, size=0.8, raster=True)
 + geom_hline(yintercept=0, color=ORANGE, size=1)
 + scale_y_continuous(limits=(-1_500_000, 1_500_000),
                      labels=lambda v: [f"${x/1e6:g}M" for x in v])
 + scale_x_continuous(limits=(0, 6000))
 + labs(x="Building square footage", y="Residual (actual − predicted)")
 + theme_minimal())

<!-- slide 38 -->
## Which Assumptions Hold?

Look back at the residual plot.

- Does the spread of the residuals stay constant? (**Equal variability**)
- Is there a curve left over that a line missed? (**Linearity**)
- Could two sales be related, e.g. neighbors on the same block? (**Independence**)
- What could we change: a log transformation (Week 5), more predictors, a different model?

The fan is the Week 5 "cone shape" again, and the pattern OpenIntro (IMS §24.6) calls "the most common pattern observed when this condition fails: the variability of y is larger when x is larger." So equal variability fails. A log transform of price (Week 5 log-linearize slide) often stabilizes this. Independence is about the data design: sales in the same neighborhood share unmeasured factors.

<!-- slide 39 -->
# Multiple Linear Regression

<!-- slide 40 -->
## Bedrooms Alone: A Line

In [ ]:
def fit(cols):
    # Design matrix: a column of 1s for the intercept, then one column per predictor
    X = np.column_stack([np.ones(homes.height)] + [homes[c].to_numpy() for c in cols])
    # lstsq returns the thetas with the smallest sum of squared residuals
    return np.linalg.lstsq(X, y, rcond=None)[0]

theta = fit(["bedrooms"])
print(f"bedrooms only:   {theta[1]:+,.0f} per bedroom")

In [ ]:
(ggplot(homes, aes("bedrooms", "sale_price"))
 + geom_point(color="#777777", alpha=0.08, size=0.8, raster=True,
              position=position_jitter(width=0.2, height=0, random_state=418))
 + geom_abline(intercept=theta[0], slope=theta[1], color=ORANGE, size=1.2)
 + scale_y_continuous(limits=(0, 2_000_000),
                      labels=lambda v: [f"${x/1e6:g}M" for x in v])
 + scale_x_continuous(limits=(0.5, 8.5), breaks=range(1, 9))
 + labs(x="Bedrooms", y="Sale price")
 + theme_minimal())

Alone, bedrooms stands in for size: bigger homes have more bedrooms, so the line climbs about $270k per bedroom.

<!-- slide 41 -->
## Square Footage + Bedrooms: A Plane

Figure after Timbers, Campbell, Lee, Ostblom, and Heagy, ["Multivariable Linear Regression"](https://python.datasciencebook.ca/regression2.html#multivariable-linear-regression), *Data Science: A First Introduction with Python*, §8.6, Fig. 8.7. Data: Cook County single-family sales, 2025; plane fit on all sales, 300 shown. Box drawn like the Week 5 Barocas figures.

In [ ]:
theta = fit(["building_sqft", "bedrooms"])
print(f"sqft + bedrooms: {theta[1]:+,.0f} per sqft, {theta[2]:+,.0f} per bedroom")

<img src="../assets/week8/house-plane-3d.svg" style="max-height: 320px;" alt="Three-dimensional box with bedrooms and building square footage along the floor and sale price as the height, holding 300 Cook County home sales as dots. A shaded orange plane climbs steeply in the square footage direction and stays nearly level in the bedrooms direction; dots below the plane show faintly through it.">

. . .

Why does the bedroom slope collapse once square footage is in the model?

One predictor fits a line; two fit a plane. This one climbs with square footage and barely tilts with bedrooms. Holding square footage fixed, an extra bedroom means the same space split into more rooms, so the slope drops from about +$270k to about −$17k. The sign is not trustworthy: drop homes over 4,000 sq ft and it turns slightly positive (about +$6k). That is the instability Lau, Gonzalez, and Nolan and Timbers et al. (§8.7.2, "Multicollinearity") warn about when predictors are correlated. Adding bedrooms barely improves the fit (RMSE about $392k either way). scikit-learn's LinearRegression gives the same thetas; DSFAI §8.6 runs this exact model (sqft + beds) on Sacramento homes. DSFAI recommends against 3D charts in general and uses one "only to illustrate what the regression plane looks like for learning purposes." Same here: with three or more predictors there is no picture, only the equation.

<!-- slide 42 -->
## Multiple Linear Regression

Sources: Lau, Gonzalez, and Nolan, ["Multiple Linear Model"](https://learningds.org/ch/15/linear_multi.html), *Learning Data Science*, §15.4; Timbers, Campbell, Lee, Ostblom, and Heagy, ["Multivariable Linear Regression"](https://python.datasciencebook.ca/regression2.html#multivariable-linear-regression), *Data Science: A First Introduction with Python*, §8.6.

"The multiple linear model ... uses more than one feature to predict (or describe or explain) the outcome." (Lau, Gonzalez, and Nolan)

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p$$

- Same three steps: model, average squared loss, minimize (now over $p + 1$ parameters).
- With two predictors, the predictions "form a flat plane" instead of a line. (Timbers et al.)
- Read each slope in the presence of the others: "Holding fixed the values of the other variables in the model ..., an increase of 1 unit in $x$ corresponds to a $\theta_1$ change in $y$, on average." (Lau, Gonzalez, and Nolan)
- "When two explanatory features are highly correlated, their coefficients in the model may be unstable." (Lau, Gonzalez, and Nolan)

<!-- slide 43 -->
## Any questions?

<!-- TODO: add this week's deadlines, as in Week 6 -->
- PrairieLearn Homeworks?
- Group Projects?
- Other logistics?
- (You can also stay after class for questions)

<!-- slide 44 -->
## Upload your worksheet

Take a picture of your worksheet, upload it in Canvas

<!-- slide 45 -->
# That's All for Today, See You Wednesday!

<!-- slide 46 -->
# Week 8, Day 2

<h2>Intro to Classification</h2>

Jack Bandy
2026

<!-- slide 47 -->
## Map

- Week 8: Roosevelt
- Day 2
    - Where We Left Off
    - When Regression Doesn't Fit
    - Intro to Classification
    - Four Classifiers: A Preview
    - Training, Validation, and Test Sets

![](../assets/orange-line-map-stops/stop09-roosevelt.svg)

<!-- slide 48 -->
# Where We Left Off

<!-- slide 49 -->
## The Modeling Process, So Far

1. Define the model: $\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2$
2. Define the loss: average squared loss (MSE)
3. Minimize the loss: find $\hat{\theta}$
- The response $y$ was sale price, a number on a continuous scale.
- So every prediction $\hat{y}$ was a number too: a price.
- Overfitting: a model can fit the data in hand and still predict new data badly.
<img class="figure" src="../assets/week8/house-plane-3d.svg" alt="Three-dimensional box with bedrooms and building square footage along the floor and sale price as the height, holding 300 Cook County home sales as dots, with a shaded orange regression plane through them.">

<!-- slide 50 -->
## Two Kinds of Supervised Learning

Hal Daumé III, [*A Course in Machine Learning*](http://ciml.info/), v0.99, ch. 1 ("Decision Trees").

"At a basic level, machine learning is about predicting the future based on the past." (Daumé)

- **Regression**: predict a real value (e.g., sale price, % tip).
- **Classification**: predict a categorical value.
    - Binary: a yes/no response (e.g., spam or not spam).
    - Multiclass: one of several classes (e.g., country of origin).
- Both start from examples where we already know the answer, which is what makes them *supervised*.

<!-- slide 51 -->
# When Regression Doesn't Fit

<!-- slide 52 -->
## Public or Private?

A school has 20,000 undergraduates and charges \$14,000 in in-state tuition and fees. Public or private?

<img src="../assets/week8/cls-universities-scatter-plain-no-uic.svg" style="max-height: 440px;" alt="Scatter plot of 1,717 colleges, undergraduate enrollment against in-state tuition and fees, with every point in the same gray and no classes or schools marked yet.">

<img src="../assets/week8/cls-universities-scatter-plain.svg" style="max-height: 440px;" alt="Scatter plot of 1,717 colleges, undergraduate enrollment against in-state tuition and fees, with every point in the same gray and no classes shown yet. UIC is marked near 22,000 undergraduates and 14,000 dollars.">

<img src="../assets/week8/cls-universities-scatter.svg" style="max-height: 440px;" alt="Scatter plot of 1,717 U.S. colleges, undergraduate enrollment against in-state tuition and fees, both on log scales. Public schools, orange triangles, form a band from about 5,000 to 20,000 dollars in tuition and from a few hundred to 60,000 students. Private nonprofits, gray circles, sit above them at 20,000 to 60,000 dollars, plus a tail of small, cheaper colleges to the left. UIC is marked among the public schools, near 22,000 undergraduates and 14,000 dollars.">

Data: U.S. Department of Education, [College Scorecard](https://collegescorecard.ed.gov/data/), most-recent institution file (June 2026 release): 1,717 predominantly bachelor's-granting schools, public or private nonprofit, that report in-state tuition and fees.

<!-- slide 53 -->
## Spam or Not?

Data: Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt, [Spambase](https://doi.org/10.24432/C53G6X), UCI Machine Learning Repository (1999), CC BY 4.0. 1,200 of 4,601 emails shown.

An email is 1% exclamation points, and its runs of capital letters average 5 long. Spam?

<img src="../assets/week8/cls-spam-scatter-plain.svg" style="max-height: 440px;" alt="Scatter plot of 1,200 emails, exclamation-point share against average run of capital letters, with every point in the same gray and no classes shown yet.">

<img src="../assets/week8/cls-spam-scatter.svg" style="max-height: 440px;" alt="Scatter plot of 1,200 emails, the share of characters that are exclamation points against the average run of capital letters, both on log scales. About half the emails have no exclamation points and form a vertical stripe at zero, mixing spam and not spam. Among the rest, spam, orange triangles, sits higher and further right than non-spam, gray circles.">

<!-- slide 54 -->
## Adélie or Gentoo Penguin?

Data: Kristen B. Gorman, Tony D. Williams, and William R. Fraser, [*PLoS ONE*](https://doi.org/10.1371/journal.pone.0090081) 9(3): e90081 (2014), via Allison Horst, Alison Hill, and Kristen Gorman, [palmerpenguins](https://doi.org/10.5281/zenodo.3960218) (2020), CC0. 274 penguins near Palmer Station, Antarctica, 2007–2009.

A penguin's bill is 45 mm long and 15 mm deep. Adélie or Gentoo?

<img src="../assets/week8/cls-penguins-scatter-plain.svg" style="max-height: 440px;" alt="Scatter plot of 274 penguins, bill length against bill depth, with every point in the same gray and no species shown yet. Two clusters are already visible.">

<img src="../assets/week8/cls-penguins-scatter.svg" style="max-height: 440px;" alt="Scatter plot of 151 Adélie and 123 Gentoo penguins, bill length against bill depth in millimeters. Adélie, gray circles, cluster at short, deep bills (about 39 mm long, 18 mm deep). Gentoo, orange triangles, cluster at long, shallow bills (about 47 mm long, 15 mm deep). The two clusters do not overlap.">

<!-- slide 55 -->
## Malignant or Benign?

Data: William Wolberg, Olvi Mangasarian, Nick Street, and W. Street, [Breast Cancer Wisconsin (Diagnostic)](https://doi.org/10.24432/C5DW2B), UCI Machine Learning Repository (1993), CC BY 4.0. 569 biopsies: 357 benign, 212 malignant.

A biopsy's cell nuclei have a mean radius of 15 and a mean texture of 22. Malignant or benign?

<img src="../assets/week8/cls-tumors-scatter-plain.svg" style="max-height: 440px;" alt="Scatter plot of 569 biopsies, mean nucleus radius against mean texture, with every point in the same gray and no diagnoses shown yet.">

<img src="../assets/week8/cls-tumors-scatter.svg" style="max-height: 440px;" alt="Scatter plot of 569 breast biopsies, mean radius of the cell nuclei against mean texture. Benign samples, gray circles, sit at smaller radii, mostly 8 to 15. Malignant samples, orange triangles, sit at larger radii, mostly 15 to 25, and slightly higher texture. The groups overlap between radius 13 and 17.">

<!-- slide 56 -->
## Chicago or the Suburbs?

Data: Cook County single-family sales, 2025 (Day 1), 1,000 of 24,764 shown, with lot size from the Assessor; 2020 census tract housing units and land area, U.S. Census Bureau [TIGERweb](https://tigerweb.geo.census.gov/).

A home on a 5,000-square-foot lot sold in a census tract with 3,000 housing units per square mile. Chicago or the suburbs?

<img src="../assets/week8/cls-cook-county-scatter-plain.svg" style="max-height: 420px;" alt="Scatter plot of 1,000 Cook County home sales, lot size against housing units per square mile in the tract, with every point in the same gray and no city or suburb labels shown yet.">

<img src="../assets/week8/cls-cook-county-scatter.svg" style="max-height: 420px;" alt="Scatter plot of 1,000 Cook County home sales, lot size in square feet against housing units per square mile in the sale's census tract, both on log scales. Chicago sales, orange triangles, crowd the upper left: lots of about 3,000 to 5,000 square feet in tracts with 3,000 to 10,000 housing units per square mile. Suburban sales, gray circles, spread to the lower right, with larger lots and sparser tracts, but many suburban sales overlap the Chicago cluster.">

<!-- slide 57 -->
## Five Yes/No Questions

Data for all five: [`datasets/classification-examples`](https://github.com/jackbandy/data-science-fun/tree/main/datasets/classification-examples) in the course repo, with sources in its README.

Each has two numeric inputs and one yes/no output.

| Question | Inputs | Data |
|---|---|---|
| Public or private university? | undergraduate enrollment, in-state tuition | College Scorecard |
| Spam or not? | "!" frequency, runs of CAPITALS | UCI Spambase |
| Adélie or Gentoo penguin? | bill length, bill depth | Palmer Penguins |
| Malignant or benign tumor? | cell nucleus radius, texture | UCI Breast Cancer Wisconsin |
| Chicago or the suburbs? | lot size, housing units per square mile | Cook County home sales, 2020 census |

<!-- slide 58 -->
## A Decision Boundary

Data: College Scorecard, as above. Boundary from a linear support vector machine fit to both inputs (more on Week 9 Wednesday).

<img class="figure" src="../assets/week8/cls-universities-boundary.svg" alt="The enrollment and in-state tuition scatter plot of public and private schools, now split by a straight diagonal boundary. The lower-right region, shaded light orange, is predicted public; the upper-left region, shaded light gray, is predicted private. Nearly all orange triangles fall on the orange side and nearly all gray circles on the gray side; the mistakes sit near the line.">
- A classifier splits the input space into regions, one label per region.
- The line where the label changes is the **decision boundary**.
- Points on the wrong side are mistakes: this line gets 93 of 1,717 schools wrong.
- Different classifiers draw different boundaries from the same data.

<!-- slide 59 -->
# Intro to Classification

<!-- slide 60 -->
## What is classification?
Classification is the task of predicting the class label of an observation from a set of predictor variables.

- The response variable is categorical rather than continuous.
- Predictor variables may be numeric, categorical, or a mixture.
- The model is fitted on training observations with known labels.
- Each new observation is assigned to one of the defined classes.

![](../assets/week8/cls-universities-boundary.svg)

<p class="code-caption" style="margin-top:0.5em;">Two features </p>

<!-- slide 61 -->
## The Same Three Steps

- Input: training examples $\{(\mathbf{x}^{(1)}, y^{(1)}), \ldots, (\mathbf{x}^{(n)}, y^{(n)})\}$ from an unknown distribution.
- Model: a hypothesis $h$ that maps features to a class, $h: X \to Y$.
- Loss: compare the true class $y$ with the prediction $h(\mathbf{x})$. The simplest is **0-1 loss**:

    $$\ell(y, h(\mathbf{x})) = \begin{cases} 0 & \text{if } y = h(\mathbf{x}) \\ 1 & \text{otherwise} \end{cases}$$

- Averaged over the data, 0-1 loss is the **error rate**: the portion of examples the classifier gets wrong.
- Minimize: find the $h$ with the lowest error, on future examples, not only the training data.

<!-- slide 62 -->
# Four Classifiers: A Preview

<!-- slide 63 -->
## Four Classifiers

| Method | How it assigns a class | More in |
|---|---|---|
| Decision tree | Asks a sequence of questions about the features, ending at a leaf with a label | Week 9, Monday |
| Naive Bayes | Picks the class with the highest probability, using Bayes' theorem | Today |
| $k$-nearest neighbors | Takes a vote among the $k$ most similar training examples | Today |
| Support vector machine | Draws the separating line (hyperplane) with the widest margin | Week 9, Wednesday |

<!-- TODO: Naive Bayes and kNN have no day of their own on the schedule; confirm "Today" is right or point to a later day. -->

<!-- slide 64 -->
## $k$-Nearest Neighbors

"If it walks like a duck, quacks like a duck, and looks like a duck, then it's probably a duck."

- "assigns class labels to observations based on the class labels of the $k$ 'most similar' observations"
- Choose a distance (e.g., Euclidean) and a number of neighbors $k$.
- For a new point: find its $k$ nearest training examples, take a majority vote.
- "If $k$ is too small, then the classifier may be susceptible to overfitting due to noise." Too large, and the vote includes points "that are not similar enough."
- No model to train (a "lazy learner"), but "Sensitive to the scale of the data."

<!-- slide 65 -->
## Try It

TODO

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

<!-- slide 66 -->
## Naive Bayes

a probabilistic model that uses Bayes' Theorem for classification

$$P(y \mid X) = \frac{P(X \mid y)\, P(y)}{P(X)}$$

- Example: "free" appears in 250 of 1,000 spam emails and 10 of 1,000 non-spam emails. Assume spam and non-spam are equally likely.

$$P(\text{spam} \mid \text{free}) = \frac{0.25 \times 0.5}{0.25 \times 0.5 + 0.01 \times 0.5} \approx 0.96$$

- With many features, the "naive" step assumes they are independent once the class is known: $P(X_1, \ldots, X_p \mid y) = \prod_{i=1}^{p} P(X_i \mid y)$.
- Pro: gives "probabilities that quantify the uncertainty in predictions." Con: the independence assumption "often does not hold in practice."

<!-- slide 67 -->
## Same Data, Four Boundaries

Each panel is a scikit-learn classifier fit to all the points shown: a decision tree of depth 3, Gaussian naive Bayes, $k$-nearest neighbors with $k = 15$, and a linear SVM (the last two on standardized features). Data are the same as previous slides. Made with help from LLMs.

### Universities

<img src="../assets/week8/cls-universities-four-boundaries.svg" style="max-height: 380px;" alt="Four panels, one per classifier (decision tree of depth 3, naive Bayes, k-nearest neighbors with k = 15, linear SVM), each shading the region it predicts for each class over the same scatter plot: undergraduate enrollment against in-state tuition. All four mark the low-tuition, larger schools as public: the tree with a box, naive Bayes with an oval, k-nearest neighbors with a ragged edge, and the linear SVM with one diagonal line.">

### Spam

<img src="../assets/week8/cls-spam-four-boundaries.svg" style="max-height: 380px;" alt="Four panels, one per classifier (decision tree of depth 3, naive Bayes, k-nearest neighbors with k = 15, linear SVM), each shading the region it predicts for each class over the same scatter plot: exclamation-point share against average capital-letter run. Every method marks only the lower-left corner, few exclamation points and short capital runs, as not spam, and the rest as spam; the corner is boxy for the tree, rounded for naive Bayes, ragged for k-nearest neighbors, and cut by a diagonal line for the SVM.">

### Penguins

<img src="../assets/week8/cls-penguins-four-boundaries.svg" style="max-height: 380px;" alt="Four panels, one per classifier (decision tree of depth 3, naive Bayes, k-nearest neighbors with k = 15, linear SVM), each shading the region it predicts for each class over the same scatter plot: bill length against bill depth. All four methods separate the two clusters cleanly; only the shape of the boundary between them differs: a step for the tree, a curve for naive Bayes, a slightly wiggly line for k-nearest neighbors, and a straight line for the SVM.">

### Tumors

<img src="../assets/week8/cls-tumors-four-boundaries.svg" style="max-height: 380px;" alt="Four panels, one per classifier (decision tree of depth 3, naive Bayes, k-nearest neighbors with k = 15, linear SVM), each shading the region it predicts for each class over the same scatter plot: mean nucleus radius against texture. All four methods put the boundary near a radius of 14 to 17, leaning so that higher texture tips a sample toward malignant; the tree's boundary is a set of steps, naive Bayes curves around the benign cluster, k-nearest neighbors is ragged, and the SVM is a straight line.">

### Chicago

<img src="../assets/week8/cls-cook-county-four-boundaries.svg" style="max-height: 380px;" alt="Four panels, one per classifier (decision tree of depth 3, naive Bayes, k-nearest neighbors with k = 15, linear SVM), each shading the region it predicts for each class over the same scatter plot: lot size against housing units per square mile. All four mark the small-lot, dense corner as Chicago: the tree with one box, naive Bayes with a curved region, k-nearest neighbors with a ragged region, and the linear SVM with everything above a rising diagonal line.">

Shapes to point out: the tree's boundaries are made of horizontal and vertical steps (Bello Lander's "rectilinear decision boundaries"); kNN's boundary follows the points and gets ragged where the classes overlap; the linear SVM can only draw one straight line. On the penguins all four agree, so the method matters most where the classes overlap. These are fit on every point, so they say nothing yet about how well each would do on new data; that is the last section today.

<!-- slide 68 -->
## Four Classifiers, Side by Side

| | Decision tree | Naive Bayes | $k$NN | SVM |
|---|---|---|---|---|
| Decides by | a path of feature tests | the most probable class | a vote of $k$ neighbors | the side of a hyperplane |
| Boundary | rectilinear | follows the class distributions | any shape | linear, or curved with a kernel |
| Key hyperparameter | max depth | (few) | $k$ | $C$, kernel |
| Strength | interpretable | probabilities | no training; flexible | maximum margin |
| Weakness | easy to overfit | assumes independence | slow to predict; scale-sensitive | slow to train |

<!-- slide 69 -->
## In SciKit, We all Fam

In scikit-learn, each of the four classifiers is a different object with the same `fit` and `predict`:

```python
from sklearn import tree, naive_bayes, neighbors, svm

model = tree.DecisionTreeClassifier(max_depth=3)
# model = naive_bayes.GaussianNB()
# model = neighbors.KNeighborsClassifier(n_neighbors=5)
# model = svm.SVC(C=1.0, kernel="rbf")

model.fit(X_train, y_train)      # learn from labeled examples
model.predict(X_test)            # predict classes for new ones
```

. . .

- Where do `X_train` and `X_test` come from?

<!-- slide 70 -->
# Training, Validation, and Test Sets

<!-- slide 71 -->
## Back to Overfitting

Lau, Gonzalez, and Nolan, ["Overfitting"](https://learningds.org/ch/16/ms_overfitting.html), *Learning Data Science*, §16.1.

- Day 1, Model C: a curve through nearly every point, and "new observations are not well-predicted."
- Same for classifiers
- $h$ overfits if $\text{error}_{\text{true}}(h) > \text{error}_{\text{train}}(h)$.
- Problem: "we don't know $\text{error}_{\text{true}}(h)$!"
- Standard approach: set aside a testing set from the available data, and don't look at them during training! (pretend we have not seen them).

<!-- slide 72 -->
## Parameters vs. Hyperparameters

- **Parameters** are learned from the training data.
    - Regression: the $\theta$s. Decision tree: "the nodes and branches."
- A **hyperparameter** is a "parameter that controls other parameters of the model."
    - Max tree depth, $k$ in $k$NN, $C$ in an SVM, the polynomial degree of Day 1's Model C.

<!-- slide 73 -->
## Three Sets

<div style="display:flex; gap:4px; margin:0.4em 0 0.8em; font-weight:600; color:#fff; text-align:center;">
<div style="flex:70; background:#555; padding:0.5em 0; border-radius:4px;">Training 70%</div>
<div style="flex:10; background:#f9461c; padding:0.5em 0; border-radius:4px;">Val.</div>
<div style="flex:20; background:#222; padding:0.5em 0; border-radius:4px;">Test 20%</div>
</div>

- Training set: Used to learn the parameters of the ML model
- Validation set (tuning/development/held-out data): can use to learn hyperparameter(s)
- Test set: Used to evaluate how well we're doing on new unseen examples.

<!-- slide 74 -->
## How to Split

- How? Randomly, Temporally, Geographically... Depends on application (often random).
- What size? "Typically between 75%-25% and 90%-10%."
    - "Larger training set → more complex models"
    - "Larger test set → better estimate of generalization error"
- Which would you use for daily CTA ridership? For Cook County home sales?

<!-- slide 75 -->
## Cross-Validation

Cross validation simulates multiple train test-splits on the training data.

<div style="display:grid; grid-template-columns: auto repeat(5, 1fr); gap:4px; max-width:720px; margin:0.4em 0; font-size:0.7em; font-weight:600; text-align:center;">
<div style="text-align:right; padding-right:0.4em;">Split 1</div><div style="background:#f9461c; color:#fff;">V</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div>
<div style="text-align:right; padding-right:0.4em;">Split 2</div><div style="background:#555; color:#fff;">T</div><div style="background:#f9461c; color:#fff;">V</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div>
<div style="text-align:right; padding-right:0.4em;">Split 3</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#f9461c; color:#fff;">V</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div>
<div style="text-align:right; padding-right:0.4em;">Split 4</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#f9461c; color:#fff;">V</div><div style="background:#555; color:#fff;">T</div>
<div style="text-align:right; padding-right:0.4em;">Split 5</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#555; color:#fff;">T</div><div style="background:#f9461c; color:#fff;">V</div>
</div>

- 5-fold: cut the training data into 5 folds; each fold takes one turn as the validation set (V) while the rest train (T).
- Pick the hyperparameter setting with the best average validation error across folds.
- "You can only use the test dataset once after deciding on the model."

<!-- slide 76 -->
## The Full Workflow

1. "Split your data into training and test sets (e.g., 80%-20%)"
2. "Use only the training data when designing, training, and tuning the model"
    - use cross-validation to choose hyperparameters; "Do not look at the test data"
3. "Commit to your final model and train once more using only the training data (including validation)."
4. "Test the final model using the test data."
5. "Train on all available data and ship it!"

<!-- slide 77 -->
## Any questions?

- PrairieLearn Homeworks?
- Group Projects?
- Other logistics?
- (You can also stay after class for questions)

<!-- slide 78 -->
## Upload your worksheet

Take a picture of your worksheet, upload it in Canvas

<!-- slide 79 -->
# That's All for Today, See You Monday!

<!-- slide 80 -->
# Appendix: Bias and Variance

<!-- slide 81 -->
## Bias and Variance

Source: Scott Fortmann-Roe, ["Understanding the Bias-Variance Tradeoff"](http://scott.fortmann-roe.com/docs/BiasVariance.html) (2012). Figure redrawn after his bulls-eye diagram.

- Consider each dot as one model
- The orange center is the target
- **Bias**: "the difference between the expected (or average) prediction of our model and the correct value which we are trying to predict."
- **Variance**: "the variability of a model prediction for a given data point."
<img class="figure" src="../assets/week8/bias-variance-bullseye.svg" alt="Four bulls-eye targets in a two-by-two grid. Columns are low and high variance; rows are low and high bias. Low bias, low variance: dots clustered on the center. Low bias, high variance: dots scattered around the center. High bias, low variance: dots clustered tightly but off to the upper right. High bias, high variance: dots scattered and off-center.">

<!-- slide 82 -->
## The Bias-Variance Tradeoff

Source: Scott Fortmann-Roe, ["Understanding the Bias-Variance Tradeoff"](http://scott.fortmann-roe.com/docs/BiasVariance.html) (2012), after Hastie, Tibshirani, and Friedman, *The Elements of Statistical Learning* (2009); Lau, Gonzalez, and Nolan, ["Model Bias and Variance"](https://learningds.org/ch/16/ms_risk.html), *Learning Data Science*, §16.6.

$$\text{Err}(x) = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

- Irreducible error: "the noise term in the true relationship that cannot fundamentally be reduced by any model."
- "Underfitting with too simple a model can lead to high model bias ..., and overfitting with too complex a model can result in high model variance." (Lau, Gonzalez, and Nolan)
- common tradeoff between minimizing the bias and minimizing the variance

<!-- slide 83 -->
# Appendix: The Big Data Paradox

<!-- slide 84 -->
## The Big Data Paradox

Source: Valerie C. Bradley, Shiro Kuriwaki, Michael Isakov, Dino Sejdinovic, Xiao-Li Meng, and Seth Flaxman, ["Unrepresentative Big Surveys Significantly Overestimated US Vaccine Uptake"](https://doi.org/10.1038/s41586-021-04198-4), *Nature* 600 (2021): 695–700. Via [Stephen Wild on Bluesky](https://bsky.app/profile/stephenjwild.bsky.social/post/3mxlur7jeo22c).

TODO

- **Discuss the big data paradox**: more data shrinks variance, not bias.
- "Increasing data size shrinks confidence intervals but magnifies the effect of survey bias: an instance of the Big Data Paradox."
- Delphi–Facebook (about 250,000 responses per week) overestimated May 2021 vaccine uptake by 17 percentage points; an Axios–Ipsos panel with about 1,000 per week was reliable.
- "Our central message is that data quality matters more than data quantity."

<!-- slide 85 -->
# Sources

1. GitHub source: <https://github.com/jackbandy/data-science-fun/blob/main/docs/slides/week8.qmd>.
1. Last modified and compiled October 13, 16:59h Central (Chicago) time.
2. Slides developed using materials from [Elena Zheleva](https://www.cs.uic.edu/~elena/) and [Gonzalo Bello Lander](https://cs.uic.edu/profiles/gonzalo-bello/), the Berkeley DS 100 team, Marine Carpuat, and Brian Ziebart.
3. Slide deck built with [Quarto](https://quarto.org/) revealjs.
4. Title font is Big Shoulders; Body font is [Libre Franklin](https://en.wikipedia.org/wiki/Franklin_Gothic#Libre_Franklin).
5. Sam Lau, Joey Gonzalez, and Deb Nolan, *[Learning Data Science](https://learningds.org/)* (O'Reilly, 2023), ch. 4 ("Modeling with Summary Statistics"), ch. 15 ("Linear Models"), §16.1 ("Overfitting"), and §16.6 ("Model Bias and Variance").
6. Scott Fortmann-Roe, ["Understanding the Bias-Variance Tradeoff"](http://scott.fortmann-roe.com/docs/BiasVariance.html) (June 2012). The bulls-eye figure is redrawn after his.
7. George E. P. Box and Norman R. Draper, *Empirical Model-Building and Response Surfaces* (Wiley, 1987), p. 424.
8. David J. C. MacKay, *[Sustainable Energy — Without the Hot Air](https://www.withouthotair.com/)* (UIT Cambridge, 2009), technical ch. A, ["Cars II"](https://www.withouthotair.com/cA/page_254.shtml).
9. R. Quinlan, [Auto MPG](https://doi.org/10.24432/C5859H) [Dataset], UCI Machine Learning Repository (1993), CC BY 4.0.
10. Data 100 course notes, *[Principles and Techniques of Data Science](https://ds100.org/course-notes/)*, UC Berkeley: ["Modeling & SLR"](https://ds100.org/course-notes/modeling-slr) and ["Constant Model, Loss, and Transformations"](https://ds100.org/course-notes/loss-transformations).
11. Mine Çetinkaya-Rundel and Johanna Hardin, *[Introduction to Modern Statistics](https://openintro-ims.netlify.app/)*, 2nd ed. (OpenIntro), §7.1.3 and §24.6.
12. Tiffany Timbers, Trevor Campbell, Melissa Lee, Joel Ostblom, and Lindsey Heagy, *[Data Science: A First Introduction with Python](https://python.datasciencebook.ca/)*, ch. 8 ("Regression II: Linear Regression"), §8.6 and §8.7.
13. Penn State, [STAT 462 §6.1, "MLR Model Assumptions"](https://online.stat.psu.edu/stat462/node/145/), for the alternate LINE names on the practice figures.
14. LINE practice data: Cook County Assessor's Office home sales (2025); Olympic 100 m winning times (Tatem et al., *Nature* 431:525, 2004, plus Wikipedia for 2008–2024); airline passengers, 1949–1960 (Box and Jenkins, *Time Series Analysis*, 1976); CTA ridership and Chicago building footprints (City of Chicago Data Portal); US presidential elections (Data 100 `elections.csv`); IMDb non-commercial datasets (June 2026). Details in the repo's [`assets/linear-model-criteria-assumptions/SOURCES.md`](https://github.com/jackbandy/data-science-fun/blob/main/docs/assets/linear-model-criteria-assumptions/SOURCES.md).
15. Gonzalo Bello Lander, CS 418 Lecture 09 ("Introduction to Data Modeling") and Lecture 10 ("Regression (I): Simple Linear Regression"), 2026; Elena Zheleva, CS 418 lecture slides, Feb. 12 and 17, 2026.
16. Gonzalo Bello Lander, CS 418 Lecture 12 ("Classification (I): Basic Concepts & Decision Trees") and Lecture 13 ("Classification (II): More Classification Techniques"), 2026; Elena Zheleva, CS 418 lecture slides, Feb. 19, Feb. 24, Feb. 26, and Mar. 3, 2026.
17. Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)*, v0.99, ch. 1 ("Decision Trees").
18. scikit-learn developers, [API reference](https://scikit-learn.org/stable/api/index.html), for the classifier class names on the "In SciKit, We all Fam" slide.
19. Classification examples (Day 2), via the repo's [`datasets/classification-examples`](https://github.com/jackbandy/data-science-fun/tree/main/datasets/classification-examples): U.S. Department of Education, [College Scorecard](https://collegescorecard.ed.gov/data/) (June 2026 release); Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt, [Spambase](https://doi.org/10.24432/C53G6X), UCI Machine Learning Repository (1999), CC BY 4.0; Allison Horst, Alison Hill, and Kristen Gorman, [palmerpenguins](https://doi.org/10.5281/zenodo.3960218) (2020), CC0, with data from Kristen B. Gorman, Tony D. Williams, and William R. Fraser, [*PLoS ONE*](https://doi.org/10.1371/journal.pone.0090081) 9(3): e90081 (2014); William Wolberg, Olvi Mangasarian, Nick Street, and W. Street, [Breast Cancer Wisconsin (Diagnostic)](https://doi.org/10.24432/C5DW2B), UCI Machine Learning Repository (1993), CC BY 4.0; U.S. Census Bureau, [TIGERweb](https://tigerweb.geo.census.gov/) 2020 census tracts.